# 01. 환경 설정 · 데이터 탐색 · 전처리

> **SageMaker AI 워크샵 (Track A: 빌트인 XGBoost)** — 학생 중도탈락 예측

이 워크샵에서는 학생의 인적사항·성적·거시경제 지표로 **졸업 / 중도탈락 / 재학**을 예측하는
다중 클래스 분류 모델을 SageMaker로 처음부터 끝까지 만들어 봅니다.

## 이 노트북에서 배우는 것
- SageMaker 세션 · 실행 역할(Execution Role) · 기본 S3 버킷 설정
- 원본 데이터를 S3에 업로드
- 간단한 탐색적 데이터 분석(EDA)과 **클래스 불균형** 확인
- **SageMaker Processing Job**으로 전처리(스케일링 없이 라벨 인코딩 + 분할) 수행

## 진행 방식
곳곳에 `____` 로 표시된 **빈칸**과 `# TODO` 주석이 있습니다. 힌트와 공식 문서 링크를 참고해
직접 채워 보세요. 막히면 강사용 `solutions/` 노트북의 같은 셀을 참고하면 됩니다.

> 이 노트북은 **SageMaker Studio JupyterLab**의 Python 3 커널(SageMaker Python SDK 포함
> 이미지, 예: *Data Science* / *Base Python*)에서 실행하는 것을 전제로 합니다.


## 0. 환경 설정

> 아래 셀은 SageMaker Python SDK **v2** 를 설치(또는 고정)합니다. 이 워크샵의 코드는 v2 API
> (`Estimator`, `HyperparameterTuner`, `Transformer` 등)에 맞춰져 있습니다.
> **커널을 재시작할 필요 없이** 그대로 다음 셀을 실행하면 됩니다.

In [ ]:
# SageMaker Python SDK v2 설치 (이미 v2라면 무시됨, v3가 설치되어 있으면 다운그레이드)
# Studio에서 커널의 작업 디렉토리가 유실될 수 있어 먼저 복구합니다.
import os, subprocess, sys
try:
    os.getcwd()
except OSError:
    os.chdir(os.path.expanduser("~"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "sagemaker>=2,<3", "-qU"],
                      stdout=subprocess.DEVNULL)
import sagemaker
print(f"SageMaker SDK v{sagemaker.__version__} ready")

In [ ]:
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
region = sess.boto_region_name

# TODO: SageMaker 실행 역할(IAM Role)과 기본 버킷을 가져오세요.
#  - 힌트: get_execution_role() 은 현재 노트북에 연결된 IAM 역할을 반환합니다.
#  - 힌트: sess.default_bucket() 은 계정/리전별 기본 버킷 이름을 반환(없으면 생성)합니다.
#  - 참고: https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning-ex-role.html
role = ____
bucket = ____

# 버킷이 존재하지 않으면 생성합니다 (default_bucket()은 이름만 반환할 수 있습니다).
import boto3
s3 = boto3.client("s3", region_name=region)
try:
    s3.head_bucket(Bucket=bucket)
except Exception:
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket)
    else:
        s3.create_bucket(Bucket=bucket, CreateBucketConfiguration={"LocationConstraint": region})
    print(f"버킷 생성 완료: {bucket}")

prefix = "sm-workshop-students"
print("region :", region)
print("role   :", role)
print("bucket :", bucket)
print("prefix :", prefix)

## 1. 원본 데이터 S3 업로드

리포지토리 루트의 `dataset/data.csv` 를 S3에 올립니다. (노트북 작업 디렉토리는 `notebooks/` 이므로 상대경로는 `../dataset/...` 입니다.)

In [ ]:
# TODO: 로컬 데이터셋을 S3에 업로드하고, 반환되는 S3 URI를 raw_s3 에 저장하세요.
#  - 힌트: sess.upload_data(path=..., bucket=..., key_prefix=...) 는 업로드된 S3 URI를 돌려줍니다.
#  - key_prefix 는 f"{prefix}/raw" 를 사용하세요.
#  - 참고: https://sagemaker.readthedocs.io/en/v2.219.0/api/utility/session.html
raw_s3 = ____
print("raw data:", raw_s3)

## 2. 탐색적 데이터 분석 (EDA)

In [ ]:
import pandas as pd

# TODO: data.csv 는 세미콜론(;)으로 구분되어 있습니다. 올바른 구분자로 읽으세요.
#  - 참고: pandas.read_csv 의 sep 인자
df = pd.read_csv("../dataset/data.csv", sep=____)

# 컬럼명 끝에 섞여 있는 공백/탭 문자를 정리합니다(실무 데이터는 지저분합니다!).
df.columns = [c.strip() for c in df.columns]
print("shape:", df.shape)
df.head()

In [ ]:
# 타깃 분포 확인
counts = df["Target"].value_counts()
ratio = df["Target"].value_counts(normalize=True).round(3)
summary = pd.concat([counts, ratio], axis=1)
summary.columns = ["count", "ratio"]
print(summary)

ax = counts.plot(kind="bar", title="Target distribution", rot=0)
ax.set_xlabel("class"); ax.set_ylabel("count")

### 관찰
- 타깃은 **3개 클래스**: `Graduate`(≈50%), `Dropout`(≈32%), `Enrolled`(≈18%) — **불균형**이 있습니다.
- 따라서 정확도(accuracy)만으로 성능을 판단하면 소수 클래스(`Enrolled`)를 놓치기 쉽습니다.
  뒤의 평가 노트북에서 **macro-F1**과 **클래스별 지표**를 함께 보는 이유입니다.
- 모든 피처가 이미 수치형이라 별도 인코딩/스케일링 없이도 트리 기반 모델(XGBoost)에 바로 넣을 수 있습니다.
  전처리에서는 **타깃 라벨 인코딩**과 **학습/검증/테스트 분할**에 집중합니다.

## 3. 전처리 스크립트 작성

SageMaker Processing Job은 **별도 컨테이너**에서 스크립트를 실행합니다. 아래 셀은 `%%writefile`
매직으로 `src/preprocessing.py` 파일을 생성합니다.

> **빌트인 XGBoost 입력 규칙**: 학습/검증 CSV는 **헤더가 없어야 하며, 첫 번째 열이 정답(label)** 이어야 합니다.
> 이 규칙을 스크립트에서 지키는 것이 핵심 포인트입니다.

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/preprocessing.py
import os
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT = "/opt/ml/processing/input/data.csv"
LABEL_MAP = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}


def main():
    # TODO: 세미콜론 구분자로 읽으세요.
    df = pd.read_csv(INPUT, sep=____)
    df.columns = [c.strip() for c in df.columns]

    # TODO: 문자열 타깃을 정수로 인코딩하세요. (힌트: .str.strip().map(LABEL_MAP))
    df["Target"] = ____

    y = df["Target"]
    X = df.drop(columns=["Target"])

    # TODO: 클래스 비율을 유지하도록 stratify 옵션을 지정하세요. (70/15/15 분할)
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=____)
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=____)

    for name in ["train", "validation", "test"]:
        os.makedirs(f"/opt/ml/processing/{name}", exist_ok=True)

    # 빌트인 XGBoost용: label을 맨 앞 열로, 헤더 없이 저장
    train = pd.concat([y_train, X_train], axis=1)
    val = pd.concat([y_val, X_val], axis=1)
    # TODO: 학습/검증 CSV를 header 없이, index 없이 저장하세요.
    train.to_csv("/opt/ml/processing/train/train.csv", header=____, index=False)
    val.to_csv("/opt/ml/processing/validation/validation.csv", header=____, index=False)

    # 평가(배치 변환)용: 피처만(라벨 제외) / 정답 라벨은 따로 저장
    X_test.to_csv("/opt/ml/processing/test/test_x.csv", header=False, index=False)
    y_test.to_csv("/opt/ml/processing/test/test_y.csv", header=False, index=False)
    print("preprocessing done:", train.shape, val.shape, X_test.shape)


if __name__ == "__main__":
    main()

## 4. SageMaker Processing Job 실행

`SKLearnProcessor` 는 scikit-learn이 설치된 관리형 컨테이너에서 위 스크립트를 실행합니다.

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

processed = f"s3://{bucket}/{prefix}/processed"

# TODO: SKLearnProcessor를 생성하세요.
#  - framework_version="1.2-1", instance_type="ml.m5.xlarge", instance_count=1
#  - role=role, sagemaker_session=sess, base_job_name="student-preprocess"
#  - 참고: https://sagemaker.readthedocs.io/en/v2.219.0/frameworks/sklearn/sagemaker.sklearn.html
sklearn_processor = SKLearnProcessor(
    framework_version=____,
    role=role,
    instance_type=____,
    instance_count=1,
    base_job_name="student-preprocess",
    sagemaker_session=sess,
)

sklearn_processor.run(
    code="src/preprocessing.py",
    # TODO: 원본 데이터를 컨테이너의 /opt/ml/processing/input 으로 매핑하세요.
    inputs=[ProcessingInput(source=raw_s3, destination=____)],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train", destination=f"{processed}/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation", destination=f"{processed}/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test", destination=f"{processed}/test"),
    ],
)
print("done")

## 5. 결과 경로 저장

다음 노트북들이 사용할 S3 경로를 `%store` 로 저장합니다.

In [ ]:
train_s3 = f"s3://{bucket}/{prefix}/processed/train/train.csv"
validation_s3 = f"s3://{bucket}/{prefix}/processed/validation/validation.csv"
test_x_s3 = f"s3://{bucket}/{prefix}/processed/test/test_x.csv"
test_y_s3 = f"s3://{bucket}/{prefix}/processed/test/test_y.csv"

%store bucket prefix region train_s3 validation_s3 test_x_s3 test_y_s3
print("stored:")
for k in ["bucket", "prefix", "region", "train_s3", "validation_s3", "test_x_s3", "test_y_s3"]:
    print(" ", k, "=", eval(k))

✅ **완료!** 전처리된 데이터가 S3에 저장되었습니다. 다음은 `02_training.ipynb` — 빌트인 XGBoost로 모델을 학습합니다.